# ЛР-03: Бюджет топливного резерва

## Worked example: military 02

Это полностью разобранный example по duality и анализу чувствительности. Он показывает полный цикл от прямой модели до сценарных пересчётов.

## 1. Исходный кейс

Здесь duality помогает понять, чего не хватает для устойчивого резерва топлива.

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 95 |
| Персонал | 58 |
| Ёмкости хранения | 47 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Северный резерв | 87 | 36 | 17 | 16 |
| Южный резерв | 83 | 32 | 15 | 14 |
| Мобильные топливозаправщики | 76 | 24 | 18 | 11 |
| Автоматизация учёта топлива | 69 | 18 | 8 | 9 |

## 2. Как читать двойственную задачу в этом примере

Прямая задача выбирает масштабы программ `x_j`: какие действия взять и насколько. Двойственная задача смотрит на ту же ситуацию как в зеркале: она назначает внутренние оценки ограничениям.

- `y_i` - теневая цена ресурса: сколько единиц эффекта даёт одна дополнительная единица бюджета, трудозатрат или операционной ёмкости.
- `z_j` - теневая цена верхней границы `x_j <= 1`: насколько ценно было бы разрешить программе стать больше 100 процентов.
- Нулевая теневая цена не означает, что ресурс бесполезен вообще. Она означает, что в текущем оптимуме добавление именно этого ресурса локально не улучшает результат.

Единица измерения shadow price всегда смешанная: `единицы эффекта / единица соответствующего ограничения`. Поэтому бюджетная теневая цена, трудовая теневая цена и теневая цена операционной ёмкости измеряются по-разному.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog


def solve_primal(effects, A_ub, b_ub, bounds):
    """Решает прямую задачу максимизации через `linprog`.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ в целевой функции.
        A_ub (np.ndarray): Матрица расхода ресурсов по программам.
        b_ub (np.ndarray): Вектор доступных лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных `0 <= x <= 1`.

    Возвращает:
        tuple: Пара `result, shadow_prices`, где `result` — ответ solver-а,
        а `shadow_prices` — теневые цены ресурсных ограничений.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    c = -effects
    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=bounds,
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)

    shadow_prices = -result.ineqlin.marginals
    return result, shadow_prices


def solve_dual(effects, A_ub, b_ub):
    """Решает двойственную задачу для модели с верхними границами.

    Аргументы:
        effects (np.ndarray): Вектор эффектов прямой задачи.
        A_ub (np.ndarray): Матрица ресурсных ограничений прямой задачи.
        b_ub (np.ndarray): Вектор правых частей ресурсных ограничений.

    Возвращает:
        scipy.optimize.OptimizeResult: Решение двойственной LP-модели.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    resource_count, program_count = A_ub.shape
    upper_bound_costs = np.ones(program_count)

    c_dual = np.concatenate(
        [
            b_ub,
            upper_bound_costs,
        ]
    )
    A_dual = -np.hstack(
        [
            A_ub.T,
            np.eye(program_count),
        ]
    )
    b_dual = -effects
    dual_bounds = [(0, None)] * (resource_count + program_count)

    result = linprog(
        c_dual,
        A_ub=A_dual,
        b_ub=b_dual,
        bounds=dual_bounds,
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)

    return result


def rerun_with_resource_change(effects, A_ub, b_ub, bounds, resource_index, delta):
    """Пересчитывает прямую задачу после изменения одного ресурса.

    Аргументы:
        effects (np.ndarray): Вектор эффектов программ.
        A_ub (np.ndarray): Матрица расхода ресурсов.
        b_ub (np.ndarray): Исходный вектор лимитов ресурсов.
        bounds (list[tuple[float, float]]): Границы переменных прямой задачи.
        resource_index (int): Индекс ресурса, лимит которого меняется.
        delta (float): Приращение правой части выбранного ограничения.

    Возвращает:
        tuple: Новый вектор лимитов и результат повторного решения модели.

    Исключения:
        RuntimeError: Если solver не смог найти оптимальное решение.
    """

    new_b = b_ub.copy()
    new_b[resource_index] += delta
    result, _ = solve_primal(effects, A_ub, new_b, bounds)

    return new_b, result

In [2]:
# Шаг 1. Задаем читаемые подписи, вектор эффектов и ресурсную матрицу.

program_names = [
    'Северный резерв',
    'Южный резерв',
    'Мобильные топливозаправщики',
    'Автоматизация учёта топлива',
]

resource_names = [
    'Бюджет',
    'Персонал',
    'Ёмкости хранения',
]

effects = np.array(
    [
        87,
        83,
        76,
        69,
    ],
    dtype=float,
)

A_ub = np.array(
    [
        [36, 32, 24, 18],
        [17, 15, 18, 8],
        [16, 14, 11, 9],
    ],
    dtype=float,
)

b_ub = np.array(
    [
        95,
        58,
        47,
    ],
    dtype=float,
)

bounds = [(0, 1)] * len(effects)

# Шаг 2. Показываем данные в табличном виде перед решением.

effects_df = pd.DataFrame(
    {
        "программа": program_names,
        "эффект на единицу": effects,
    }
)
A_ub_df = pd.DataFrame(
    A_ub,
    index=resource_names,
    columns=program_names,
)
b_ub_df = pd.DataFrame(
    {
        "ресурс": resource_names,
        "лимит": b_ub,
    }
)

print("Вектор эффектов:")
display(effects_df)

print("Матрица ресурсных коэффициентов A_ub:")
display(A_ub_df)

print("Вектор правых частей b_ub:")
display(b_ub_df)

# Шаг 3. Решаем прямую и двойственную задачи, затем проверяем сильную двойственность.
primal_result, shadow_prices = solve_primal(effects, A_ub, b_ub, bounds)
dual_result = solve_dual(effects, A_ub, b_ub)
strong_duality_ok = np.allclose(-primal_result.fun, dual_result.fun)
assert strong_duality_ok

plan_df = pd.DataFrame(
    {
        "программа": program_names,
        "x*": np.round(primal_result.x, 4),
        "эффект на единицу": effects,
    }
)
resources_df = pd.DataFrame(
    {
        "ресурс": resource_names,
        "лимит": b_ub,
        "slack": np.round(primal_result.slack, 4),
        "shadow_price": np.round(shadow_prices, 4),
        "binding": np.isclose(primal_result.slack, 0.0),
    }
)

print("Оптимальный эффект (primal):", round(-primal_result.fun, 4))
print("Оптимальное значение dual:", round(dual_result.fun, 4))
print("Сильная двойственность проверена:", strong_duality_ok)
print()
print("Оптимальный план:")
display(plan_df)

print("Ресурсный разбор:")
display(resources_df)

Вектор эффектов:


,программа,эффект на единицу
0,Северный резерв,87.0
1,Южный резерв,83.0
2,Мобильные топливозаправщики,76.0
3,Автоматизация учёта топлива,69.0


Матрица ресурсных коэффициентов A_ub:


,Северный резерв,Южный резерв,Мобильные топливозаправщики,Автоматизация учёта топлива
Бюджет,36.0,32.0,24.0,18.0
Персонал,17.0,15.0,18.0,8.0
Ёмкости хранения,16.0,14.0,11.0,9.0


Вектор правых частей b_ub:


,ресурс,лимит
0,Бюджет,95.0
1,Персонал,58.0
2,Ёмкости хранения,47.0


Оптимальный эффект (primal): 278.75
Оптимальное значение dual: 278.75
Сильная двойственность проверена: True

Оптимальный план:


,программа,x*,эффект на единицу
0,Северный резерв,0.5833,87.0
1,Южный резерв,1.0000,83.0
2,Мобильные топливозаправщики,1.0000,76.0
3,Автоматизация учёта топлива,1.0000,69.0


Ресурсный разбор:


,ресурс,лимит,slack,shadow_price,binding
0,Бюджет,95.0,0.0000,2.4167,True
1,Персонал,58.0,7.0833,0.0000,False
2,Ёмкости хранения,47.0,3.6667,0.0000,False


## 3. Интерпретация после ресурсной таблицы

В таблице ресурсов строка с `binding = True` показывает ограничение, которое держит оптимум. Для него `slack` близок к нулю, а `shadow_price` показывает локальную предельную ценность ресурса.

Если `shadow_price = 0`, это нулевая теневая цена: ресурс в данной оптимальной точке не является узким местом. Если значение положительно, одна дополнительная единица измерения этого ресурса должна примерно увеличить оптимальный эффект на величину `shadow_price`.

Прогноз по теневой цене локален. Его корректно читать для малых изменений правой части `b_ub`, а затем проверять повторным решением модели.

In [3]:
# Шаг 4. Сравниваем прогноз по теневой цене с повторным решением.

scenario_specs = [
    ('Бюджет +3', 0, 3),
    ('Ёмкости хранения +2', 2, 2),
]

scenario_rows = []
for label, resource_index, delta in scenario_specs:
    new_b, new_result = rerun_with_resource_change(
        effects,
        A_ub,
        b_ub,
        bounds,
        resource_index,
        delta,
    )
    predicted = shadow_prices[resource_index] * delta
    actual = (-new_result.fun) - (-primal_result.fun)
    scenario_rows.append(
        {
            "сценарий": label,
            "ресурс": resource_names[resource_index],
            "delta": delta,
            "прогноз по shadow price": round(predicted, 4),
            "факт после пересчёта": round(actual, 4),
            "разница": round(actual - predicted, 4),
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

,сценарий,ресурс,delta,прогноз по shadow price,факт после пересчёта,разница
0,Бюджет +3,Бюджет,3,7.25,7.25,0.0
1,Ёмкости хранения +2,Ёмкости хранения,2,0.00,0.00,0.0


In [4]:
# Шаг 5. Проверяем отдельный сценарий изменения коэффициента цели.
objective_label = 'Топливозаправщики +7 к эффекту'
objective_index = 2
objective_delta = 7

new_effects = effects.copy()
new_effects[objective_index] += objective_delta
objective_result, _ = solve_primal(new_effects, A_ub, b_ub, bounds)
objective_df = pd.DataFrame(
    {
        "сценарий": [objective_label],
        "новый оптимальный эффект": [round(-objective_result.fun, 4)],
        "изменение эффекта": [round((-objective_result.fun) - (-primal_result.fun), 4)],
        "новое значение программы": [round(objective_result.x[objective_index], 4)],
    }
)

display(objective_df)

,сценарий,новый оптимальный эффект,изменение эффекта,новое значение программы
0,Топливозаправщики +7 к эффекту,285.75,7.0,1.0


## 4. Что важно проговорить в выводе

- какие ограничения оказались binding и почему именно они держат оптимум;
- какой ресурс имеет наибольшую теневую цену и что это означает содержательно;
- в какой единице измерения записана каждая теневая цена;
- почему нулевая теневая цена не означает бесполезность ресурса вообще;
- чем ресурсные dual-переменные `y_i` отличаются от dual-переменных верхних границ `z_j`;
- почему прогноз по shadow price является локальным;
- насколько хорошо совпал прогноз по shadow price с фактическим пересчётом;
- меняется ли структура плана при небольших изменениях `b` и `c`.